In [1]:
from pathlib import Path
import json

from app.schemas import GenerationProfile, QuestionDepth, GeneratedQA
from app.rag.embedding import EmbeddingManager
from app.rag.vector_store import VectorStore
from app.rag.retriever import Retriever
from app.rag.reranker import Reranker
from app.generation.qa_generator import QAGenerator

/Users/macstudio/Desktop/Development/DomainForge V1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path.cwd()
VECTORSTORE_DIR = BASE_DIR / "data" / "vectorstore"
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "generated_qa.jsonl"

In [3]:
embed_mgr = EmbeddingManager(model_name="BAAI/bge-base-en-v1.5")
vector_store = VectorStore(
    persist_dir=VECTORSTORE_DIR,
    collection_name="domainforge_governance",
    embedding_manager=embed_mgr
)

retriever = Retriever(vector_store=vector_store, top_k=4)
print(f"Vector Store Connection Successful: {vector_store.count()} records found.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7668.32it/s]


Vector Store Connection Successful: 2347 records found.


In [4]:

seed_query = "What are the core technical characteristics, risk management controls, and accountability principles in trustworthy AI?"
candidate_contexts = retriever.retrieve(seed_query, top_k=20)

reranker = Reranker(model_name="BAAI/bge-reranker-base")
top_5_contexts = reranker.rerank(seed_query, candidate_contexts, top_n=5)

profile = GenerationProfile(question_count=2, depth=QuestionDepth.COMPARATIVE)
generator = QAGenerator(model_name="deepseek-r1:14b")

synthesized_dataset = generator.generate(top_5_contexts, profile)

for i, qa in enumerate(synthesized_dataset, 1):
    print(f"\n--- Synthesized Q/A #{i} ---")
    print(f"Question   : {qa.question}")
    print(f"Answer     : {qa.answer}")
    print("Sources:")
    for src in qa.sources:
        print(f"  • {src.source} (Page {src.page}) [{src.chunk_id}]")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 32710.79it/s]



--- Synthesized Q/A #1 ---
Question   : How do transparency and accountability mechanisms in AI systems contribute to building trust, and what are the key principles that ensure these mechanisms are effective?
Answer     : Transparency and accountability are foundational to building trust in AI systems. Transparency involves clear communication of AI purposes, data use, and decision-making processes, ensuring users understand how systems operate. Accountability requires mechanisms to assign responsibility and ensure that AI systems comply with established norms and regulations. Key principles include clarity in communication, accessibility of information, and consistent application of governance frameworks. These principles ensure that transparency and accountability are not just theoretical but are practically implemented and verifiable.
Sources:
  • NIST.AI.100-1.pdf (Page 5) [NIST.AI.100-1_p5_c14]
  • NIST.AI.100-1.pdf (Page 17) [NIST.AI.100-1_p17_c10]
  • NIST.AI.100-1.pdf (Page 1

Previous Response:

--- Synthesized Q/A #1 ---

Question   : How do the characteristics of trustworthy AI systems, such as transparency and accountability, compare across different governance frameworks like NIST and the EU's AI Act?

Answer     : The characteristics of trustworthy AI systems, including transparency and accountability, are foundational across various governance frameworks. NIST emphasizes these principles by outlining specific controls and mechanisms to ensure AI systems are transparent in their decision-making processes and accountable in their outcomes. Similarly, the EU's AI Act also prioritizes transparency by requiring AI systems to provide clear explanations of their operations and decisions. While NIST focuses on technical controls and standards, the EU's AI Act introduces regulatory requirements, such as mandatory risk assessments and governance frameworks. Both approaches, however, harmonize on the need for robust governance mechanisms to ensure trustworthiness, with NIST providing detailed implementation guidelines and the EU focusing on legal enforceability.

Sources:

  • NIST.AI.100-1.pdf (Page 5) [NIST.AI.100-1_p5_c14]

  • NIST.AI.100-1.pdf (Page 17) [NIST.AI.100-1_p17_c10]

  • NIST.AI.100-1.pdf (Page 17) [NIST.AI.100-1_p17_c4]

  • NIST.AI.600-1.pdf (Page 16) [NIST.AI.600-1_p16_c11]

  • NIST.AI.600-1.pdf (Page 13) [NIST.AI.600-1_p13_c12]



--- Synthesized Q/A #2 ---

Question   : What mechanisms do governance frameworks like NIST suggest for ensuring the continuous improvement of AI systems' reliability and safety?

Answer     : NIST suggests several mechanisms for continuous improvement of AI systems' reliability and safety, including regular monitoring, evaluation, and iterative updates based on feedback. These mechanisms align with principles from other frameworks, such as the EU's AI Act, which also emphasizes continuous risk assessment and mitigation. NIST's approach is particularly detailed, providing a structured approach to identifying and addressing vulnerabilities through systematic testing and validation processes. By integrating these mechanisms, frameworks like NIST ensure that AI systems remain reliable and safe over their lifecycle, adapting to new challenges and evolving technologies.

Sources:

  • NIST.AI.100-1.pdf (Page 5) [NIST.AI.100-1_p5_c14]

  • NIST.AI.100-1.pdf (Page 17) [NIST.AI.100-1_p17_c10]

  • NIST.AI.100-1.pdf (Page 17) [NIST.AI.100-1_p17_c4]

  • NIST.AI.600-1.pdf (Page 16) [NIST.AI.600-1_p16_c11]

  • NIST.AI.600-1.pdf (Page 13) [NIST.AI.600-1_p13_c12] 